# Lab 1 — NumPy: shape, broadcasting và vectorization

**Thời lượng:** 75–90 phút  
**Mục tiêu:** dự đoán shape/dtype trước khi chạy; dùng boolean mask, broadcasting và vectorization; benchmark có kiểm soát.

> Quy tắc: trước mỗi cell, hãy viết output dự đoán vào một cell Markdown riêng.

In [ ]:
import platform
import timeit

import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)
print('Python:', platform.python_version())
print('NumPy:', np.__version__)

## 1. Anatomy của ndarray

Dự đoán `shape`, `ndim`, `size`, `dtype` và kết quả giảm theo từng axis.

In [ ]:
x = np.arange(24, dtype=np.float64).reshape(2, 3, 4)
print('shape:', x.shape)
print('ndim:', x.ndim)
print('size:', x.size)
print('dtype:', x.dtype)
print('sum axis=0 shape:', x.sum(axis=0).shape)
print('sum axis=1 shape:', x.sum(axis=1).shape)
print('sum axis=2 shape:', x.sum(axis=2).shape)
assert x.shape == (2, 3, 4)
assert x.sum(axis=-1).shape == (2, 3)

## 2. Indexing, boolean mask, view và copy

In [ ]:
prices = np.array([12.5, 55.0, 8.0, 120.0, 33.5, 75.0])
selected = prices[(prices >= 20) & (prices <= 80)]
discounted = prices.copy()
discounted[discounted > 50] *= 0.9
print('selected:', selected)
print('original:', prices)
print('discounted copy:', discounted)
np.testing.assert_allclose(selected, [55.0, 33.5, 75.0])
np.testing.assert_allclose(prices, [12.5, 55.0, 8.0, 120.0, 33.5, 75.0])

In [ ]:
base = np.arange(6)
view = base[1:4]
view[0] = 999
print('base after editing view:', base)

base2 = np.arange(6)
independent = base2[1:4].copy()
independent[0] = 999
print('base2 after editing copy:', base2)
assert base[1] == 999
assert base2[1] == 1

## 3. Broadcasting

Ma trận dưới đây có shape `(4, 3)`. Mean và std theo cột có shape `(3,)`, nên broadcast theo chiều cuối.

In [ ]:
features = np.array([
    [1.0, 10.0, 100.0],
    [2.0, 20.0, 120.0],
    [3.0, 30.0, 140.0],
    [4.0, 40.0, 160.0],
])
mean = features.mean(axis=0)
std = features.std(axis=0)
z = (features - mean) / std
print('features:', features.shape)
print('mean:', mean.shape)
print('std:', std.shape)
print('z:', z.shape)
np.testing.assert_allclose(z.mean(axis=0), 0.0, atol=1e-12)
np.testing.assert_allclose(z.std(axis=0), 1.0, atol=1e-12)

In [ ]:
row_mean_wrong_shape = features.mean(axis=1)
try:
    features - row_mean_wrong_shape
except ValueError as exc:
    print('Expected broadcasting error:', exc)

row_mean = features.mean(axis=1, keepdims=True)
centered_by_row = features - row_mean
print('row_mean shape:', row_mean.shape)
np.testing.assert_allclose(centered_by_row.mean(axis=1), 0.0, atol=1e-12)

## 4. Vectorized revenue

In [ ]:
quantity = np.array([2, 1, 4, 3])
unit_price = np.array([10.0, 50.0, 2.5, 20.0])
discount_pct = np.array([0.0, 10.0, 20.0, 5.0])
gross = quantity * unit_price
net = gross * (1 - discount_pct / 100)
print('gross:', gross)
print('net:', net)
np.testing.assert_allclose(gross, [20.0, 50.0, 10.0, 60.0])
np.testing.assert_allclose(net, [20.0, 45.0, 8.0, 57.0])

## 5. Controlled benchmark

Không dùng lần nhanh nhất. Warm-up, đo nhiều lần, báo median và xác nhận hai output tương đương.

In [ ]:
n = 200_000
q = rng.integers(1, 6, size=n)
p = rng.uniform(1, 200, size=n)
d = rng.choice([0.0, 5.0, 10.0, 20.0], size=n)

def loop_revenue(quantity, price, discount):
    output = []
    for q_i, p_i, d_i in zip(quantity, price, discount):
        output.append(q_i * p_i * (1 - d_i / 100))
    return np.asarray(output)

def vector_revenue(quantity, price, discount):
    return quantity * price * (1 - discount / 100)

loop_output = loop_revenue(q[:100], p[:100], d[:100])
vector_output = vector_revenue(q[:100], p[:100], d[:100])
np.testing.assert_allclose(loop_output, vector_output)

loop_times = timeit.repeat(lambda: loop_revenue(q, p, d), repeat=5, number=1)
vector_times = timeit.repeat(lambda: vector_revenue(q, p, d), repeat=5, number=1)
loop_median = float(np.median(loop_times))
vector_median = float(np.median(vector_times))
print(f'loop median: {loop_median:.6f}s')
print(f'vector median: {vector_median:.6f}s')
print(f'speed ratio: {loop_median / vector_median:.1f}x')
assert vector_median < loop_median

## 6. Exit ticket

Viết câu trả lời của bạn:

1. Broadcasting rule bằng ngôn ngữ của bạn.
2. Một trường hợp nên dùng `.copy()`.
3. Vì sao benchmark trên array nhỏ có thể gây hiểu sai?
4. Một phép toán bạn từng viết loop nhưng giờ có thể vectorize.

In [ ]:
print('Lab 1 checks: PASS')